In [ ]:
import evaluate
from transformers import T5Tokenizer, MT5ForConditionalGeneration
import pandas as pd
from tqdm import tqdm
import torch

In [ ]:
metric = evaluate.load("sacrebleu")

In [ ]:
MODELNAME = "./models/gec_german_mt5"
MAX_LENGTH = 128

tokenizer = T5Tokenizer.from_pretrained(MODELNAME, legacy=False, use_fast=True)
model = MT5ForConditionalGeneration.from_pretrained(MODELNAME)

In [ ]:
df = pd.read_json("./data/chapter1-eval-v4.jsonl", lines=True)
data_corrupted = df["de_corrupted"].str.strip()
data_correct = df["de_correct"].str.strip()

data_corrupted = data_corrupted.tolist()#[:20]
data_correct = data_correct.tolist()#[:20]

In [ ]:
tokenized_inputs = tokenizer(
    data_corrupted,
    max_length=MAX_LENGTH,
    truncation=True,
    padding="max_length",
    return_tensors="pt"
)

In [ ]:
tokenized_inputs['input_ids'].shape

In [ ]:
if torch.cuda.is_available():
    model.to('cuda')
    tokenized_inputs = {k: v.to('cuda') for k, v in tokenized_inputs.items()}

# call in batches of 8
BATCH_SIZE = 8
all_outputs = []
for i in tqdm(range(0, len(data_corrupted), BATCH_SIZE)):
    batch_input_ids = tokenized_inputs['input_ids'][i:i+BATCH_SIZE]
    batch_attention_mask = tokenized_inputs['attention_mask'][i:i+BATCH_SIZE]
    
    batch_output = model.generate(
        input_ids=batch_input_ids,
        attention_mask=batch_attention_mask,
        max_length=MAX_LENGTH,
        num_beams=5, 
        early_stopping=True,
        repetition_penalty=2.5
    )
    
    all_outputs.extend(batch_output)

# corrected_text = tokenizer.decode(output[0], skip_special_tokens=True)
# corrected_text

# for token_id in output[0]:
#     token = tokenizer.decode([token_id])
#     print(f"{token_id.item()}: '{token}'")

In [ ]:
decoded_preds = [tokenizer.decode(ids, skip_special_tokens=True) for ids in all_outputs]
for s1,s2 in zip(decoded_preds, data_correct):
    print(f"{s1}: '{s2}'")

In [ ]:
result = metric.compute(predictions=decoded_preds, references=data_correct)
result